In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

# import arviz as az
from jax.random import PRNGKey

# import numpyro
# from numpyro import distributions as dist
# from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.graph import *
from summer3.epi import *

In [ ]:
location = Stratification("location", ["North", "South"])
humans = CompartmentMap.new(location)
age_strat = humans.stratify(Stratification("age", ["child", "adult", "older"]))
cities = humans.stratify(Stratification("city", ["A","B"]), location["North"])

#disease_state = Stratification("disease_state", ["S", "I", "R"])
#humans = humans.stratify(disease_state)

In [ ]:
age_strat.categories().product(location.categories())

In [ ]:
times = pd.date_range("1 jan 1980", "1 jan 1981")
epi_model = CompartmentalEpiModel(humans, times)

In [ ]:
proc_vals_ref = np.random.normal(0.0, 0.01, len(times))

def random_proc(t, proc_vals):
    cproc_vals = jnp.cumsum(proc_vals)
    return jnp.exp(jnp.interp(t, np.arange(len(proc_vals)), cproc_vals, left=cproc_vals[0], right=cproc_vals[-1]))

rp_cgf = defer(random_proc)(Time, Parameter("proc_vals", proc_vals_ref))

In [ ]:
reloc_S_to_N = TransitionFlow(
    "reloc_S_to_N",
    location["South"],
    location["North"],
    Parameter("reloc_rate", 0.1) * rp_cgf,
)

reloc_N_to_S = TransitionFlow(
    "reloc_N_to_S",
    location["North"],
    location["South"],
    Parameter("reloc_rate", 0.1) * rp_cgf,
)

In [ ]:
epi_model.add_flow(reloc_S_to_N)
epi_model.add_flow(reloc_N_to_S)
epi_model.set_initial_population(
    base_pops=age_strat.categories().wrap(np.array([3000, 2000, 1000])),
    pop_splits=[location.categories().wrap(np.array([0.1, 0.9]))],
)

In [ ]:
def skew_by_age(age_skew: float = 0.0) -> CategoryData:
    rates = jnp.exp(jnp.linspace(0.0 - age_skew, age_skew, len(age_strat.strata)))
    return age_strat.categories().wrap(rates)

In [ ]:
reloc_S_to_N.

In [ ]:
epi_model.flows["reloc_S_to_N"].adjustments_source.append(
    defer(skew_by_age)(Parameter("age_skew_SN", 0.0))
)

epi_model.flows["reloc_N_to_S"].adjustments_dest.append(
    defer(skew_by_age)(Parameter("age_skew_NS", 0.0))
)

In [ ]:
def city_adjustment(p_a, p_b) -> CategoryData:
    return cities.categories().wrap(jnp.array([p_a, p_b]))

In [ ]:
epi_model.flows["reloc_N_to_S"].adjustments_source.append(
    defer(city_adjustment)(Parameter("city_skew_A", 0.0), Parameter("city_skew_B", 0.0))
)

In [ ]:
import diffrax as dfx
stepsize_controller = dfx.PIDController(
                rtol=1e-5, atol=1e-5, dtmax=7.0
            )

adjoint = dfx.RecursiveCheckpointAdjoint(checkpoints=1024)

solver_kwargs = {
    "stepsize_controller": stepsize_controller,
    "adjoint": adjoint,
}

In [ ]:
params = {"reloc_rate": 0.001, "age_skew_SN": -2.0, "age_skew_NS": -0.2, "city_skew_A": 0.6, "city_skew_B": 0.9, "proc_vals": proc_vals_ref}
results = epi_model.run(params, solver_kwargs=solver_kwargs)

In [ ]:
type(results["aux"])

In [ ]:
def extract_data(results):
    return results["compartments"]#.sumcats(compartment=location.categories())

In [ ]:
ref_data = extract_data(results)

In [ ]:
def mdata(params) -> ManagedArray:
    modelled_res = epi_model.run(params, solver_kwargs=solver_kwargs)
    modelled_data = extract_data(modelled_res)#modelled_res["flows"]["reloc_N_to_S"].sumcats(source=age_strat.categories())
    return modelled_data

In [ ]:
@jit
def loss(params) -> jax.Array:
    #modelled_res = epi_model.run(params, solver_kwargs=solver_kwargs)
    #modelled_data= modelled_res["flows"]["reloc_N_to_S"].sumcats(source=age_strat.categories())
    modelled_data = mdata(params)
    return ((ref_data.data - modelled_data.data)**2.0).sum()


In [ ]:
gloss = jit(grad(loss))


In [ ]:
pnew = params | {"reloc_rate": 0.01, "age_skew_SN": 0.0, "age_skew_NS": 0.0, "city_skew_A": 1.0, "city_skew_B": 1.0, "proc_vals": jnp.zeros(len(times))}

In [ ]:
import optax

In [ ]:
start_params = pnew

In [ ]:
start_learning_rate = 5e-3
optimizer = optax.adam(start_learning_rate)

# Initialize parameters of the model + optimizer.
#params = jnp.array([0.0, 0.0])
opt_state = optimizer.init(start_params)
cur_params = start_params
best_loss = loss(cur_params)
best_params = cur_params
print(best_loss)

In [ ]:
# A simple update loop.

print(loss(cur_params))

for _ in range(1000):
  grads = gloss(cur_params)
  updates, opt_state = optimizer.update(grads, opt_state)
  cur_params = optax.apply_updates(cur_params, updates)
  if loss(cur_params) < loss(best_params):
    best_params = cur_params

print(loss(cur_params), loss(best_params))

In [ ]:
ref_data.to_pandas_df().plot()

In [ ]:
mdata(best_params).to_pandas_df().plot()

In [ ]:
mdata(pnew).to_pandas_df().plot()

In [ ]:
pd.DataFrame(
    {
        "ref": proc_vals_ref,
        "bestp": best_params["proc_vals"]
    }
).plot()